<center><h1><strong><font color="blue"> Data Engineering for Data Science - Week 06</font></strong></h1></center>

<center><img alt="" src="images/cover.png" style="height: 600px;"/></center>

<center><h2><strong><font color="blue">Advanced Selection, Aggregations, & Normalization</font></strong></h2></center>

<b><center><h3>(C) Taufik Sutanto</h3></center>

<center><h2><strong><font color="blue">Outline</font></strong></h2></center>

**Topics**:

* DDL (Create, Alter, Drop) vs. DML (Select, Insert, Update, Delete).
* Advanced Selection: Joins (Inner, Left, Right, Full), Aggregations, and Window Functions.
* The importance of data redundancy and anomalies in research data.
* Functional Dependencies.
* Normal Forms: 1NF, 2NF, 3NF, and BCNF.
* Denormalization: When and why to break the rules.

  
**Lab**: 
* Given a flat, messy "Excel-style" dataset, decompose it into a 3NF schema.


<center><h2><strong><font color="blue"> Export-Import Data using phpMyAdmin</font></strong></h2></center>

<center><h3><strong><font color="red"> Please clone (“pull”) the course GitHub repository and locate the file "example-schema-sample-data.sql" within the "data" folder</font></strong></h3></center>

<center><img alt="" src="images/deds-04/phpmyadmin-eximport.jpg" style="height: 480px;"/></center>

<center><h2><strong><font color="red">Is this the only method for exporting and importing data to and from MySQL?</font></strong></h2></center>

<center><img alt="" src="images/curious.jpg" style="height: 320px;"/></center>

<center><img alt="" src="images/deds-05/university-registry-db-ERD.jpg" style="height: 480px;"/></center>

In [22]:
# Best Practice; use a function to connect to our database
import warnings; warnings.simplefilter('ignore')
import mysql.connector
import pandas as pd

host = "localhost"
user = "root"
password = "x1234"
database = "university_registry_db"

def connect(host="localhost", user="root", password="", database=""):
    try:
        db = mysql.connector.connect(
            host=host,
            user=user,
            password=password,
            database=database)
        con = db.cursor()
        print("Connected to the '{}' Database.".format(database))
        return db, con
    except Exception as err_:
        print("Connection to the Database server failed:")
        print(err_)

db, con = connect(host=host, user=user, password=password, database=database)

Connected to the 'university_registry_db' Database.


## Data Manipulation Language (DML) in MariaDB
While Data Definition Language (DDL) builds the "house," **Data Manipulation Language (DML)** is the act of moving furniture in. It allows researchers to interact with the actual data stored within the tables. In the context of the **CRUD** (Create, Read, Update, Delete) cycle, DML is where the majority of your daily data engineering work occurs.

## 1. The SELECT Statement (Reading Data)

The `SELECT` statement is the most frequent operation in data science. It retrieves specific records from one or more tables based on defined criteria.

### Basic Syntax
```sql
SELECT id, full_name 
FROM students 
WHERE country_origin = 'Indonesia' 
ORDER BY full_name ASC;
```

* **Pro Tip:** Avoid `SELECT *` in production code. Explicitly naming columns reduces memory overhead and prevents your Python scripts from breaking if the table schema changes later.

In [3]:
qry = """ SELECT id, full_name
FROM students 
WHERE country_origin = 'Indonesia' 
ORDER BY full_name ASC;
"""
df = pd.read_sql(qry, db) # pay attention we are using db, and not "con". Pandas recommend using alchemy SQL connection
# Also pay attention we can only use pandas "read_sql" to get some data from the database. 
# We can not execute any arbitrary query that does not get data (e.g. update or delete data)
df.head()

,id,full_name
0,2,Ayu Lestari
1,1,Budi Santoso
2,16,Siti Aminah


<center><h2><strong><font color="red">Is the query case-sensitive? Is this behavior consistent across all cases?</font></strong></h2></center>

<center><img alt="" src="images/curious.jpg" style="height: 320px;"/></center>

## 2. The INSERT Statement (Creating Data)

`INSERT` adds new rows to a table. For production-grade research, we often handle this through Python scripts to ensure data is sanitized.

### Standard Insertion
```sql
INSERT INTO instructors (full_name, email) 
VALUES ('Dr. Aris Kusuma', 'aris.k@uiii.ac.id');
```

### Multi-row Insertion
```sql
INSERT INTO students (full_name, country_origin) 
VALUES 
    ('Siti Aminah', 'Indonesia'),
    ('John Doe', 'Australia'),
    ('Li Wei', 'China');
```

In [4]:
# Good practice insert data ~ fast
qry = "INSERT INTO students (full_name, country_origin) VALUES (%s, %s)"
data = [('Siti Aminah', 'Indonesia'), ('John Doe', 'Australia'), ('Li Wei', 'China')]
data

[('Siti Aminah', 'Indonesia'), ('John Doe', 'Australia'), ('Li Wei', 'China')]

In [ ]:
con.executemany(qry, data)
db.commit() # Mandatory for data modification
print(f"Record inserted. ID: {con.lastrowid}") # you can also check from phpmyadmin

<center><h2><strong><font color="green">Essential for production pipelines</font></strong></h2></center>

<center><img alt="" src="images/recommended.jpg" style="height: 320px;"/></center>

In [5]:
import logging

try:
    con.executemany(qry, data)
    db.commit() # Mandatory for data modification
    print(f"Record inserted. ID: {con.lastrowid}")
except Exception as e:
    logging.error("Insert failed", exc_info=True)
    con.rollback()

Record inserted. ID: 19


## 3. The UPDATE Statement (Modifying Data)

The `UPDATE` statement modifies existing records. In a professional environment, this must be handled with extreme precision.

### Critical Usage
```sql
UPDATE enrollments 
SET grade = 3.85 
WHERE student_id = 101 AND course_id = 5;
```

> **Warning:** Never omit the `WHERE` clause unless you intended to change the value for every single row in the table. There is no "undo" button for a mass update in a basic configuration.

In [8]:
# Example of updating data from python

data = [(3.9, 1, 1), (2.9, 1, 2), (1.9, 2, 1)]
qry = "UPDATE enrollments SET grade = %s WHERE student_id = %s AND course_id = %s"
qry

'UPDATE enrollments SET grade = %s WHERE student_id = %s AND course_id = %s'

In [9]:
con.executemany(qry, data)
db.commit() #check on phpMyAdmin
"Done"

<center><h2><strong><font color="green">Essential Tips Insert-Update</font></strong></h2></center>

## Logic: _“update if exists, otherwise insert”_

<center><img alt="" src="images/recommended.jpg" style="height: 320px;"/></center>

```sql
INSERT INTO students (id, full_name, country_origin, enrollment_year)
VALUES (1, 'Alice', 'Indonesia', 2023)
ON DUPLICATE KEY UPDATE
    full_name = VALUES(full_name),
    country_origin = VALUES(country_origin),
    enrollment_year = VALUES(enrollment_year);
```
### Behavior:
* If id = 1 does not exist → INSERT
* If id = 1 already exists → UPDATE that row

In [12]:
qry = """
INSERT INTO students (id, full_name, country_origin, enrollment_year)
VALUES (%s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    full_name = VALUES(full_name),
    country_origin = VALUES(country_origin),
    enrollment_year = VALUES(enrollment_year)
"""
data = (1, "Siti Nurhaliza", "Indonesia", 2023)

data

(1, 'Siti Nurhaliza', 'Indonesia', 2023)

In [14]:
con.execute(qry, data)
db.commit()
con.rowcount

'Done'

## 4. The DELETE Statement (Removing Data)

* The `DELETE` statement removes specific rows from a table. Unlike the `DROP` command (which removes the table itself), `DELETE` only removes the (row) data.
* Be very careful when using `DELETE`. Deleting data in MySQL from Python is deceptively simple, but in production it’s where data loss, locking, and performance issues most often occur. 

### Targeted Deletion
```sql
DELETE FROM students WHERE id = 1
```

* **Primary key**: It is recommended to use primary key to delete data.

In [19]:
con.executemany(
    "DELETE FROM students WHERE id = %s",
    [(1,), (2,), (3,)]
)
db.commit()
"Deleted Rows = {}".format(con.rowcount)

'Affected Rows = 0'

<center><h2><strong><font color="red">NEVER Run Unbounded DELETE (Critical)</font></strong></h2></center>

<center><img alt="" src="images/warning.jpg" style="height: 320px;"/></center>

<center><h3><strong><font color="blue">e.g. "DELETE FROM students;"</font></strong></h3></center>

## DML Command Lifecycle

| Command | CRUD Equivalent | Research Utility |
| :--- | :--- | :--- |
| **SELECT** | Read | Extracting features for ML models or EDA. |
| **INSERT** | Create | Ingesting sensor data or web-scraped results. |
| **UPDATE** | Update | Cleaning "dirty" data or updating model logs. |
| **DELETE** | Delete | Removing outliers or outdated experimental runs. |

<center><h2><strong><font color="blue">Understanding MariaDB Character Set and Collation</font></strong></h2></center>

In database management, these two terms are often grouped together, but they handle very different parts of how your data is processed. Think of it as the difference between **knowing which letters exist** and **knowing how to alphabetize them**.

## 1. Character Set (The "What")
A **Character Set** is a defined list of characters and their corresponding binary values. It determines **what** specific symbols, letters, and numbers the database is capable of storing.

* **Function:** It maps a character (like 'A' or '😊') to a specific number (like 65 or a 4-byte Unicode sequence) that the computer understands.
* **Common Example:** `utf8mb4`. This set is widely used because it can store almost every character from every language, plus emojis and mathematical symbols.

## 2. Collation (The "How")
A **Collation** is a set of rules used to **compare and sort** the characters within a specific character set. It determines the logic used in `ORDER BY` clauses or when checking equality in a `WHERE` clause.

* **Function:** It defines whether the database should be case-sensitive (is 'A' the same as 'a'?) and how to handle accents (is 'e' the same as 'é'?).
* **Common Example:** `utf8mb4_unicode_ci`. The `_ci` at the end stands for **Case Insensitive**, meaning the database treats 'Apple' and 'apple' as identical when searching or sorting.

## 3. Key Differences at a Glance

| Feature | Character Set | Collation |
| :--- | :--- | :--- |
| **Primary Goal** | Storage and encoding. | Sorting and comparison. |
| **Responsibility** | Defines *which* characters are valid. | Defines *how* valid characters relate to each other. |
| **Analogy** | The symbols in an alphabet. | The rules in a dictionary for sorting words. |
| **Scope** | Determines disk space (e.g., 1-byte vs 4-byte). | Determines query results (e.g., sorting order). |

## 4. Why the Distinction Matters
If you choose the wrong **Character Set**, you might end up with "broken" characters (often appearing as `?` or ``) because the database doesn't recognize the symbol you're trying to save. 

If you choose the wrong **Collation**, your data will be stored correctly, but your search results might be unexpected—for instance, a search for "resume" might fail to find "Résumé," or your alphabetical list might put lowercase 'z' before uppercase 'A'.

<center><h2><strong><font color="blue">Understanding MariaDB Character Set and Collation</font></strong></h2></center>

<center><img alt="" src="images/deds-04/DB-collations-character-sets.png" style="height: 600px;"/></center>

### Related Queries

```sql
CREATE DATABASE research_db 
CHARACTER SET utf8mb4 
COLLATE utf8mb4_unicode_ci;
-----------------------------------
CREATE TABLE international_students (
    id INT PRIMARY KEY,
    student_name VARCHAR(100)
) CHARACTER SET utf8mb4 COLLATE utf8mb4_bin;
-----------------------------------
SHOW TABLE STATUS LIKE 'students';
```

<center><h2><strong><font color="blue">The MariaDB my.ini Configuration File</font></strong></h2></center>

<center><img alt="" src="images/deds-04/mysql-my-ini-file.png" style="height: 600px;"/></center>

<center><h2><strong><font color="blue">Prologue/Motivation ~ Normalization</font></strong></h2></center>

<center><img alt="" src="images/deds-05/motivation-normalization.png" style="height: 600px;"/></center>

# Database Design: Redundancy, Anomalies, and Functional Dependencies

Efficient database design is critical for maintaining data integrity, especially in research environments where data accuracy and consistency are paramount. This explanation covers the pitfalls of poor schema design and the formal logic used to correct them.

## 1. Data Redundancy and Anomalies in Research Data

In a relational database, **data redundancy** occurs when the same piece of data is stored in multiple places unnecessarily. While redundant backups are good, redundancy within a schema leads to significant operational risks known as **anomalies**.

### 1.1 The Impact of Redundancy
* **Storage Inefficiency:** Large-scale research datasets (e.g., genomic data or longitudinal surveys) can become prohibitively expensive to store if duplicate information is recorded across millions of rows.
* **Data Inconsistency:** The primary danger is that one instance of a data point may be updated while others are not, leading to conflicting "truths" within the same dataset.

### 1.2 Database Anomalies
Anomalies are problems that arise during the manipulation of a poorly structured table. They are generally categorized into three types:

| Anomaly Type | Description | Research Example |
| :--- | :--- | :--- |
| **Update Anomaly** | Occurs when a change to a single data value requires multiple rows to be updated. | If a Lab Location changes, and it is stored in every row of an "Experiments" table, failing to update every row leads to inconsistent records. |
| **Insertion Anomaly** | Occurs when certain data cannot be recorded unless other, unrelated data is also provided. | You cannot record a new Researcher's profile until they are assigned to an active Experiment, because the primary key requires an Experiment ID. |
| **Deletion Anomaly** | Occurs when deleting one piece of information unintentionally results in the loss of other unrelated data. | Deleting the last Experiment entry for a specific Lab might accidentally delete all contact information for that Lab. |




## 2. Functional Dependencies (FD)

**Functional Dependency** is a constraint between two sets of attributes in a relation. It is the fundamental building block of **Normalization**, the process used to eliminate redundancy and anomalies.

### 2.1 Formal Definition
If $R$ is a relation and $X$ and $Y$ are subsets of attributes in $R$, we say $Y$ is **functionally dependent** on $X$ (denoted as $X \to Y$) if and only if each value of $X$ is associated with exactly one value of $Y$.

In simpler terms, if you know the value of $X$, you can determine the value of $Y$.

> **Example:** In a research database, `ResearcherID \to ResearcherName`. 
> If `ResearcherID` is 101, the name will always be "Dr. Smith." You won't find `ResearcherID` 101 associated with "Dr. Jones."



### 2.2 Types of Functional Dependencies

To refine database structures, we identify specific categories of FDs:

#### A. Full Functional Dependency
An attribute is fully functionally dependent if it depends on the *entire* primary key, not just a part of it.
* **Example:** In a table where the key is `{Student_ID, Course_ID}`, the `Grade` is fully dependent because you need both the student and the course to determine the grade.

#### B. Partial Functional Dependency
This occurs when an attribute depends on only a portion of a composite primary key. This is a primary cause of redundancy.
* **Example:** In the same `{Student_ID, Course_ID}` table, `Student_Name` is only dependent on `Student_ID`. Storing it here creates partial dependency.

#### C. Transitive Dependency
This occurs when a non-key attribute depends on another non-key attribute, which in turn depends on the primary key ($A \to B$ and $B \to C$).
* **Example:** `Project_ID \to Department_ID` and `Department_ID \to Department_Head`. Here, `Department_Head` has a transitive dependency on `Project_ID`.

### 2.3 Role in Normalization
Functional dependencies allow us to decompose a single, bloated table into smaller, logically sound tables:
1.  **First Normal Form (1NF):** Remove multi-valued attributes.
2.  **Second Normal Form (2NF):** Remove **Partial Dependencies**.
3.  **Third Normal Form (3NF):** Remove **Transitive Dependencies**.

By identifying $X \to Y$ relationships, researchers ensure that each fact is stored exactly once, preserving the integrity of the scientific record.

<center><img alt="" src="images/deds-05/functional-dependency.png" style="height: 600px;"/></center>

# Practice

* Start designing database schema for your thesis project

## Next Week's Discussions

* Conceptual vs. Logical vs. Physical Data Models.
* Entity-Relationship Diagrams (ERD).
* OLTP (Transactional) vs. OLAP (Analytical) systems.
* Introduction to Dimensional Modeling (Star Schema vs. Snowflake Schema).


<center><h2><strong><font color="blue">End of Module</font></strong></h2></center>

<center><img alt="" src="images/meme-cartoon/meme-db-joins.jpg" style="height: 480px;"/></center>